In [ ]:
#Ensamble learning code for four dataset meta model xgboost and mlp

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ====================== DATASETS ======================
DATASETS = {
    "LFI": "/content/drive/MyDrive/WAF_THESIS/DATASETS/lfi_cleaned.csv",
    "XSS": "/content/drive/MyDrive/WAF_THESIS/DATASETS/xss_cleaned.csv",
    "SQL": "/content/drive/MyDrive/WAF_THESIS/DATASETS/sql_cleaned.csv",
    "OSC": "/content/drive/MyDrive/WAF_THESIS/DATASETS/osc_cleaned.csv",
}

# ====================== BEST XGBOOST PARAMS ======================
XGB_PARAMS = {
    "LFI": {"n_estimators": 300, "max_depth": 5, "learning_rate": 0.1,
            "min_child_weight": 3, "gamma": 0.1, "subsample": 0.9, "colsample_bytree": 0.9},
    "XSS": {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.05,
            "min_child_weight": 1, "gamma": 0.2, "subsample": 0.8, "colsample_bytree": 1.0},
    "OSC": {"n_estimators": 400, "max_depth": 4, "learning_rate": 0.05,
            "min_child_weight": 1, "gamma": 0.2, "subsample": 1.0, "colsample_bytree": 0.8},
    "SQL": {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.1,
            "min_child_weight": 1, "gamma": 0.2, "subsample": 0.9, "colsample_bytree": 0.8},
}

# ====================== BEST MLP PARAMS ======================
MLP_PARAMS = {
    "LFI": {"lr": 0.0031428809, "dropout": 0.1849, "weight_decay": 5.34e-7, "h1": 512, "h2": 256, "h3": 128},
    "XSS": {"lr": 0.0035420771, "dropout": 0.2186, "weight_decay": 2.87e-7, "h1": 128, "h2": 64,  "h3": 32},
    "SQL": {"lr": 0.0031428809, "dropout": 0.1849, "weight_decay": 5.34e-7, "h1": 512, "h2": 256, "h3": 128},
    "OSC": {"lr": 0.0031428809, "dropout": 0.1849, "weight_decay": 5.34e-7, "h1": 512, "h2": 256, "h3": 128},
}

# ====================== MLP ARCHITECTURE ======================
class TabularMLP(nn.Module):
    def __init__(self, input_dim, h1, h2, h3, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h1, h2),        nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h2, h3),        nn.BatchNorm1d(h3), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h3, 32),        nn.ReLU(),          nn.Dropout(dropout),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x)

# ====================== HELPERS ======================
def build_xgb(ds_name):
    return XGBClassifier(**XGB_PARAMS[ds_name], random_state=SEED,
                         eval_metric="logloss", use_label_encoder=False, n_jobs=-1)

def mlp_predict_proba(model, X):
    model.eval()
    Xt = torch.tensor(X, dtype=torch.float32)
    loader = DataLoader(Xt, batch_size=256, shuffle=False)
    probs = []
    with torch.no_grad():
        for xb in loader:
            probs.extend(torch.sigmoid(model(xb.to(device))).cpu().numpy())
    return np.array(probs).squeeze()

# ====================== MAIN LOOP ======================
N_FOLDS = 5
results = []

for ds_name, ds_path in DATASETS.items():
    df = pd.read_csv(ds_path).dropna()
    X = df.iloc[:, :-1].values.astype(np.float32)
    y = df.iloc[:, -1].values.astype(int)

    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    scaler = StandardScaler()
    X_tv_sc = scaler.fit_transform(X_tv).astype(np.float32)
    X_test_sc = scaler.transform(X_test).astype(np.float32)

    # OOF Stacking
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_xgb = np.zeros(len(X_tv_sc))
    oof_mlp = np.zeros(len(X_tv_sc))
    test_xgb_folds, test_mlp_folds = [], []

    for tr_idx, val_idx in skf.split(X_tv_sc, y_tv):
        X_tr, X_val = X_tv_sc[tr_idx], X_tv_sc[val_idx]
        y_tr, y_val = y_tv[tr_idx], y_tv[val_idx]

        # Base XGBoost
        xgb = build_xgb(ds_name)
        xgb.fit(X_tr, y_tr)
        oof_xgb[val_idx] = xgb.predict_proba(X_val)[:, 1]
        test_xgb_folds.append(xgb.predict_proba(X_test_sc)[:, 1])

        # Base MLP
        p = MLP_PARAMS[ds_name]
        mlp = TabularMLP(X_tr.shape[1], p["h1"], p["h2"], p["h3"], p["dropout"]).to(device)
        opt = optim.Adam(mlp.parameters(), lr=p["lr"], weight_decay=p["weight_decay"])
        crit = nn.BCEWithLogitsLoss()

        Xtr_t = torch.tensor(X_tr, dtype=torch.float32)
        ytr_t = torch.tensor(y_tr, dtype=torch.float32)
        loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=128, shuffle=True)

        mlp.train()
        for _ in range(70):
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                crit(mlp(xb).squeeze(), yb).backward()
                opt.step()

        oof_mlp[val_idx] = mlp_predict_proba(mlp, X_val)
        test_mlp_folds.append(mlp_predict_proba(mlp, X_test_sc))

    # Meta Features
    meta_train = np.column_stack([oof_xgb, oof_mlp])
    meta_test  = np.column_stack([np.mean(test_xgb_folds, axis=0), np.mean(test_mlp_folds, axis=0)])

    meta_scaler = StandardScaler()
    meta_train_sc = meta_scaler.fit_transform(meta_train)
    meta_test_sc  = meta_scaler.transform(meta_test)

    # Meta Models
    meta_xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, random_state=SEED,
                             eval_metric="logloss", n_jobs=-1)
    meta_xgb.fit(meta_train_sc, y_tv)
    pred_xgb = meta_xgb.predict(meta_test_sc)

    meta_mlp = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu',
                             solver='adam', learning_rate_init=0.001, max_iter=300,
                             random_state=SEED)
    meta_mlp.fit(meta_train_sc, y_tv)
    pred_mlp = meta_mlp.predict(meta_test_sc)

    # Metrics
    def get_metrics(y_true, y_pred):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1": f1_score(y_true, y_pred, average="macro", zero_division=0)
        }

    m_xgb = get_metrics(y_test, pred_xgb)
    m_mlp = get_metrics(y_test, pred_mlp)


    results.append({
        "Dataset"  : ds_name,
        # XGBoost meta
        "XGB_Acc"  : round(m_xgb["Accuracy"],  4),
        "XGB_Prec" : round(m_xgb["Precision"], 4),
        "XGB_Rec"  : round(m_xgb["Recall"],    4),
        "XGB_F1"   : round(m_xgb["F1"],        4),
        # MLP meta
        "MLP_Acc"  : round(m_mlp["Accuracy"],  4),
        "MLP_Prec" : round(m_mlp["Precision"], 4),
        "MLP_Rec"  : round(m_mlp["Recall"],    4),
        "MLP_F1"   : round(m_mlp["F1"],        4),
        # Winner
        "Best"     : "XGB-meta" if m_xgb["F1"] >= m_mlp["F1"] else "MLP-meta"
    })



In [ ]:
# ====================== FINAL RESULTS ======================
results_df = pd.DataFrame(results)

print("\n" + "="*85)
print(" " * 25 + "FINAL STACKING ENSEMBLE PERFORMANCE")
print("="*85)

# === DEBUG INFO ===
print(f"\n📊 Results DataFrame Shape: {results_df.shape}")
print(f"Columns: {results_df.columns.tolist()}")
if len(results_df) == 0:
    print("⚠️  WARNING: results_df is EMPTY! No datasets were successfully processed.")
    print("Check if your dataset paths are correct and Google Drive is mounted.")
else:
    print(results_df)

# ── 1. XGBoost Meta-Model Performance ─────────────────────────────
print("\n" + "-" * 80)
print(" META-MODEL 1 : XGBoost Meta")
print("-" * 80)
print(f" {'Dataset':<10} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>10} {'F1-Score':>10} {'':>8}")
print(" " + "-" * 65)

for _, r in results_df.iterrows():
    best_flag = "← Best" if r.get("Best") == "XGB-meta" else ""
    print(f" {r.get('Dataset','N/A'):<10} {r.get('XGB_Acc',0):>10.4f} "
          f"{r.get('XGB_Prec',0):>10.4f} {r.get('XGB_Rec',0):>10.4f} "
          f"{r.get('XGB_F1',0):>10.4f} {best_flag:>8}")

# ── 2. MLP Meta-Model Performance ─────────────────────────────────
print("\n" + "-" * 80)
print(" META-MODEL 2 : MLP Meta")
print("-" * 80)
print(f" {'Dataset':<10} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>10} {'F1-Score':>10} {'':>8}")
print(" " + "-" * 65)

for _, r in results_df.iterrows():
    best_flag = "← Best" if r.get("Best") == "MLP-meta" else ""
    print(f" {r.get('Dataset','N/A'):<10} {r.get('MLP_Acc',0):>10.4f} "
          f"{r.get('MLP_Prec',0):>10.4f} {r.get('MLP_Rec',0):>10.4f} "
          f"{r.get('MLP_F1',0):>10.4f} {best_flag:>8}")

# Summary
print("\n" + "="*85)
print("OVERALL RESULTS SUMMARY")
print("="*85)
print(results_df.to_string(index=False) if not results_df.empty else "No data available")
print("\nBest Meta-Model per Dataset:")
print(results_df[['Dataset', 'Best']].to_string(index=False) if not results_df.empty else "No data")


                         FINAL STACKING ENSEMBLE PERFORMANCE

📊 Results DataFrame Shape: (4, 10)
Columns: ['Dataset', 'XGB_Acc', 'XGB_Prec', 'XGB_Rec', 'XGB_F1', 'MLP_Acc', 'MLP_Prec', 'MLP_Rec', 'MLP_F1', 'Best']
  Dataset  XGB_Acc  XGB_Prec  XGB_Rec  XGB_F1  MLP_Acc  MLP_Prec  MLP_Rec  \
0     LFI   0.9384    0.9118   0.9418  0.9250   0.9384    0.9118   0.9418   
1     XSS   0.9977    0.9967   0.9982  0.9975   0.9980    0.9972   0.9985   
2     SQL   0.9952    0.9908   0.9963  0.9935   0.9948    0.9901   0.9961   
3     OSC   0.9856    0.9843   0.9869  0.9854   0.9864    0.9850   0.9878   

   MLP_F1      Best  
0  0.9250  XGB-meta  
1  0.9978  MLP-meta  
2  0.9930  XGB-meta  
3  0.9862  MLP-meta  

--------------------------------------------------------------------------------
 META-MODEL 1 : XGBoost Meta
--------------------------------------------------------------------------------
 Dataset      Accuracy  Precision     Recall   F1-Score         
 -------------------------------

Ensamble learning stacking for marged dataset

In [ ]:
# ================================================================
#  WAF INTRUSION DETECTION — STACKING ENSEMBLE (MERGED DATASET)
#  Base Models     : XGBoost (tuned)  +  PyTorch MLP (Optuna)
#  Meta-Models     : XGBoost meta     |  MLP meta

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data        import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing   import StandardScaler
from sklearn.neural_network  import MLPClassifier
from sklearn.metrics         import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}\n")


DATASET_PATH = "/content/drive/MyDrive/WAF_THESIS/DATASETS/merged_all_attacks.csv"



# XGBoost — best params from tuning
XGB_BEST = dict(
    n_estimators     = 600,
    max_depth        = 10,
    learning_rate    = 0.05,
    gamma            = 0.3,
    min_child_weight = 1,
    subsample        = 0.9,
    colsample_bytree = 0.7,
    reg_alpha        = 0.5,
    reg_lambda       = 0,
)

# MLP — best params from Optuna
MLP_BEST = dict(
    lr           = 0.0012753663404252576,
    dropout      = 0.17385161977036334,
    weight_decay = 2.2135069349961324e-06,
    h1           = 128,
    h2           = 64,
    h3           = 128,
)


#  BASE MLP ARCHITECTURE  (Optuna tuned)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, h1, h2, h3, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1),
            nn.ReLU(), nn.Dropout(dropout),

            nn.Linear(h1, h2),        nn.BatchNorm1d(h2),
            nn.ReLU(), nn.Dropout(dropout),

            nn.Linear(h2, h3),        nn.BatchNorm1d(h3),
            nn.ReLU(), nn.Dropout(dropout),

            nn.Linear(h3, 32),        nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)



def build_xgb() -> XGBClassifier:
    """XGBClassifier with merged-dataset tuned params."""
    return XGBClassifier(
        **XGB_BEST,
        random_state = SEED,
        eval_metric  = "logloss",
        n_jobs       = -1,
    )


def train_base_mlp(X_train: np.ndarray,
                   y_train: np.ndarray,
                   epochs:  int = 70) -> TabularMLP:
    """Train PyTorch MLP base model with Optuna best params."""
    model = TabularMLP(
        X_train.shape[1],
        MLP_BEST["h1"],
        MLP_BEST["h2"],
        MLP_BEST["h3"],
        MLP_BEST["dropout"],
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr           = MLP_BEST["lr"],
        weight_decay = MLP_BEST["weight_decay"],
    )
    criterion = nn.BCEWithLogitsLoss()
    loader    = DataLoader(
        TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
        ),
        batch_size = 128,
        shuffle    = True,
    )

    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            criterion(model(xb).squeeze(), yb).backward()
            optimizer.step()

    return model


def mlp_predict_proba(model: TabularMLP,
                      X:     np.ndarray) -> np.ndarray:
    """Return sigmoid probabilities (class 1)."""
    model.eval()
    loader = DataLoader(
        torch.tensor(X, dtype=torch.float32),
        batch_size = 256,
        shuffle    = False,
    )
    probs = []
    with torch.no_grad():
        for xb in loader:
            probs.extend(
                torch.sigmoid(model(xb.to(device))).cpu().numpy()
            )
    return np.array(probs).squeeze()


def get_metrics(y_true: np.ndarray,
                y_pred: np.ndarray,
                label:  str = "") -> dict:
    acc  = accuracy_score (y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score   (y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score       (y_true, y_pred, average="macro", zero_division=0)
    if label:
        print(f"  {label:<38}  Acc={acc:.4f}  "
              f"Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}")
    return dict(Accuracy=acc, Precision=prec, Recall=rec, F1=f1)


print("=" * 65)
print("  MERGED DATASET — WAF Stacking Ensemble")
print("=" * 65)

df = pd.read_csv(DATASET_PATH).dropna()


string_cols = df.select_dtypes(include=["object"]).columns.tolist()
if string_cols:
    print(f"  Dropping string columns : {string_cols}")
    df = df.drop(columns=string_cols)

X  = df.iloc[:, :-1].values.astype(np.float32)
y  = df.iloc[:,  -1].values.astype(int)
unique, counts = np.unique(y, return_counts=True)
print(f"\n  Shape      : {df.shape}")
print(f"  Features   : {X.shape[1]}")
print(f"  Classes    : {dict(zip(unique.tolist(), counts.tolist()))}")
print(f"  Columns    : {list(df.columns)}\n")



X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler    = StandardScaler()
X_tv_sc   = scaler.fit_transform(X_tv).astype(np.float32)
X_test_sc = scaler.transform(X_test).astype(np.float32)

print(f"  Train+Val  : {X_tv_sc.shape[0]} samples")
print(f"  Test       : {X_test_sc.shape[0]} samples\n")



N_FOLDS        = 10
skf            = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_xgb        = np.zeros(len(X_tv_sc))
oof_mlp        = np.zeros(len(X_tv_sc))
xgb_test_folds = []
mlp_test_folds = []

print(f"  Building OOF meta-features ({N_FOLDS}-Fold CV) ...")
print(f"  {'Fold':<6}  {'XGB OOF F1':>12}  {'MLP OOF F1':>12}")
print("  " + "-" * 36)

for fold, (tr_idx, val_idx) in enumerate(
        skf.split(X_tv_sc, y_tv), start=1):

    X_tr,  X_val  = X_tv_sc[tr_idx], X_tv_sc[val_idx]
    y_tr,  y_val  = y_tv[tr_idx],    y_tv[val_idx]

    # ── XGBoost base ──────────────────────────────────────────
    xgb = build_xgb()
    xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb.predict_proba(X_val)[:, 1]
    xgb_test_folds.append(xgb.predict_proba(X_test_sc)[:, 1])

    # ── PyTorch MLP base ──────────────────────────────────────
    mlp = train_base_mlp(X_tr, y_tr, epochs=70)
    oof_mlp[val_idx] = mlp_predict_proba(mlp, X_val)
    mlp_test_folds.append(mlp_predict_proba(mlp, X_test_sc))

    # fold-level OOF F1
    xgb_f1 = f1_score(y_val, (oof_xgb[val_idx] >= 0.5).astype(int),
                       average="macro", zero_division=0)
    mlp_f1 = f1_score(y_val, (oof_mlp[val_idx] >= 0.5).astype(int),
                       average="macro", zero_division=0)
    print(f"  {fold:<6}  {xgb_f1:>12.4f}  {mlp_f1:>12.4f}")

# average test predictions across folds
test_xgb_avg = np.mean(xgb_test_folds, axis=0)
test_mlp_avg = np.mean(mlp_test_folds, axis=0)

# ── Base model OOF performance ────────────────────────────────
print(f"\n  Base Model OOF Performance (train-val set):")
get_metrics(y_tv, (oof_xgb >= 0.5).astype(int), "XGBoost base (OOF)")
get_metrics(y_tv, (oof_mlp >= 0.5).astype(int), "MLP base     (OOF)")



meta_train = np.column_stack([oof_xgb,      oof_mlp])
meta_test  = np.column_stack([test_xgb_avg, test_mlp_avg])

meta_scaler   = StandardScaler()
meta_train_sc = meta_scaler.fit_transform(meta_train).astype(np.float32)
meta_test_sc  = meta_scaler.transform(meta_test).astype(np.float32)



meta_xgb = XGBClassifier(
    n_estimators = 200,
    max_depth    = 3,
    learning_rate= 0.05,
    subsample    = 0.8,
    colsample_bytree = 0.8,
    random_state = SEED,
    eval_metric  = "logloss",
    n_jobs       = -1,
)
meta_xgb.fit(meta_train_sc, y_tv)
pred_xgb = meta_xgb.predict(meta_test_sc)



meta_mlp = MLPClassifier(
    hidden_layer_sizes = (32, 16),
    activation         = "relu",
    solver             = "adam",
    learning_rate_init = MLP_BEST["lr"],
    alpha              = MLP_BEST["weight_decay"],
    max_iter           = 300,
    batch_size         = 64,
    random_state       = SEED,
)
meta_mlp.fit(meta_train_sc, y_tv)
pred_mlp = meta_mlp.predict(meta_test_sc)



m_xgb = get_metrics(y_test, pred_xgb)
m_mlp = get_metrics(y_test, pred_mlp)
winner = "XGB-meta" if m_xgb["F1"] >= m_mlp["F1"] else "MLP-meta"

#  Results
print(f"\n\n{'='*65}")
print("  STACKING ENSEMBLE — TEST SET RESULTS")
print(f"{'='*65}")

print(f"\n  {'Meta-Model':<30} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>10} {'F1-Score':>10}")
print("  " + "─" * 62)
print(f"  {'XGBoost meta':<30} {m_xgb['Accuracy']:>10.4f} "
      f"{m_xgb['Precision']:>10.4f} {m_xgb['Recall']:>10.4f} "
      f"{m_xgb['F1']:>10.4f}")
print(f"  {'MLP meta':<30} {m_mlp['Accuracy']:>10.4f} "
      f"{m_mlp['Precision']:>10.4f} {m_mlp['Recall']:>10.4f} "
      f"{m_mlp['F1']:>10.4f}")
print("  " + "─" * 62)
print(f"\n  Best Meta-Model : {winner}")


print(f"\n  Classification Report — XGBoost Meta:")
print(classification_report(y_test, pred_xgb, digits=4))

print(f"  Classification Report — MLP Meta:")
print(classification_report(y_test, pred_mlp, digits=4))



results_df = pd.DataFrame([{
    "Meta_Model" : "XGBoost",
    "Accuracy"   : round(m_xgb["Accuracy"],  4),
    "Precision"  : round(m_xgb["Precision"], 4),
    "Recall"     : round(m_xgb["Recall"],    4),
    "F1_Score"   : round(m_xgb["F1"],        4),
}, {
    "Meta_Model" : "MLP",
    "Accuracy"   : round(m_mlp["Accuracy"],  4),
    "Precision"  : round(m_mlp["Precision"], 4),
    "Recall"     : round(m_mlp["Recall"],    4),
    "F1_Score"   : round(m_mlp["F1"],        4),
}])

results_df.to_csv("waf_stacking_merged_results.csv", index=False)
print(f"\n  Saved → waf_stacking_merged_results.csv")
print("=" * 65)

Device : cpu

  MERGED DATASET — WAF Stacking Ensemble
  Dropping string columns : ['attack_type']

  Shape      : (94210, 20)
  Features   : 19
  Classes    : {0: 37758, 1: 56452}
  Columns    : ['and_sign', 'sign1', 'sign2', 'sign6', 'or_sign', 'sign5', 'sign3', 'hashes', 'sign4', 'badwords', 'lessthan', 'comment_sign', 'greaterthan', 'stars', 'at_sign', 'sub_line', 'dashes', 'dolar_sign', 'exclamation', 'class']

  Train+Val  : 75368 samples
  Test       : 18842 samples

  Building OOF meta-features (10-Fold CV) ...
  Fold      XGB OOF F1    MLP OOF F1
  ------------------------------------
  1             0.9613        0.9607
  2             0.9625        0.9618
  3             0.9615        0.9615
  4             0.9612        0.9609
  5             0.9599        0.3797
  6             0.9571        0.9567
  7             0.9616        0.9610
  8             0.9560        0.9560
  9             0.9589        0.9584
  10            0.9628        0.9617

  Base Model OOF Performance

In [ ]:
#stacking ensamble model for four dataset results with logistic regrassion

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


DATASETS = {
    "LFI": "/content/drive/MyDrive/WAF_THESIS/DATASETS/lfi_cleaned.csv",
    "XSS": "/content/drive/MyDrive/WAF_THESIS/DATASETS/xss_cleaned.csv",
    "SQL": "/content/drive/MyDrive/WAF_THESIS/DATASETS/sql_cleaned.csv",
    "OSC": "/content/drive/MyDrive/WAF_THESIS/DATASETS/osc_cleaned.csv",
}


XGB_PARAMS = {
    "LFI": {"n_estimators": 300, "max_depth": 5, "learning_rate": 0.1,
            "min_child_weight": 3, "gamma": 0.1, "subsample": 0.9, "colsample_bytree": 0.9},
    "XSS": {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.05,
            "min_child_weight": 1, "gamma": 0.2, "subsample": 0.8, "colsample_bytree": 1.0},
    "OSC": {"n_estimators": 400, "max_depth": 4, "learning_rate": 0.05,
            "min_child_weight": 1, "gamma": 0.2, "subsample": 1.0, "colsample_bytree": 0.8},
    "SQL": {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.1,
            "min_child_weight": 1, "gamma": 0.2, "subsample": 0.9, "colsample_bytree": 0.8},
}


MLP_PARAMS = {
    "LFI": {"lr": 0.0031428809, "dropout": 0.1849, "weight_decay": 5.34e-7, "h1": 512, "h2": 256, "h3": 128},
    "XSS": {"lr": 0.0035420771, "dropout": 0.2186, "weight_decay": 2.87e-7, "h1": 128, "h2": 64,  "h3": 32},
    "SQL": {"lr": 0.0031428809, "dropout": 0.1849, "weight_decay": 5.34e-7, "h1": 512, "h2": 256, "h3": 128},
    "OSC": {"lr": 0.0031428809, "dropout": 0.1849, "weight_decay": 5.34e-7, "h1": 512, "h2": 256, "h3": 128},
}


class TabularMLP(nn.Module):
    def __init__(self, input_dim, h1, h2, h3, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h1, h2),        nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h2, h3),        nn.BatchNorm1d(h3), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h3, 32),        nn.ReLU(),          nn.Dropout(dropout),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x)


def build_xgb(ds_name):
    return XGBClassifier(**XGB_PARAMS[ds_name], random_state=SEED,
                         eval_metric="logloss", use_label_encoder=False, n_jobs=-1)

def mlp_predict_proba(model, X):
    model.eval()
    Xt = torch.tensor(X, dtype=torch.float32)
    loader = DataLoader(Xt, batch_size=256, shuffle=False)
    probs = []
    with torch.no_grad():
        for xb in loader:
            probs.extend(torch.sigmoid(model(xb.to(device))).cpu().numpy())
    return np.array(probs).squeeze()


N_FOLDS = 5
results = []

for ds_name, ds_path in DATASETS.items():
    df = pd.read_csv(ds_path).dropna()
    X = df.iloc[:, :-1].values.astype(np.float32)
    y = df.iloc[:, -1].values.astype(int)

    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    scaler = StandardScaler()
    X_tv_sc = scaler.fit_transform(X_tv).astype(np.float32)
    X_test_sc = scaler.transform(X_test).astype(np.float32)

    # OOF Stacking
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_xgb = np.zeros(len(X_tv_sc))
    oof_mlp = np.zeros(len(X_tv_sc))
    test_xgb_folds, test_mlp_folds = [], []

    for tr_idx, val_idx in skf.split(X_tv_sc, y_tv):
        X_tr, X_val = X_tv_sc[tr_idx], X_tv_sc[val_idx]
        y_tr, y_val = y_tv[tr_idx], y_tv[val_idx]

        # Base XGBoost
        xgb = build_xgb(ds_name)
        xgb.fit(X_tr, y_tr)
        oof_xgb[val_idx] = xgb.predict_proba(X_val)[:, 1]
        test_xgb_folds.append(xgb.predict_proba(X_test_sc)[:, 1])

        # Base MLP
        p = MLP_PARAMS[ds_name]
        mlp = TabularMLP(X_tr.shape[1], p["h1"], p["h2"], p["h3"], p["dropout"]).to(device)
        opt = optim.Adam(mlp.parameters(), lr=p["lr"], weight_decay=p["weight_decay"])
        crit = nn.BCEWithLogitsLoss()

        Xtr_t = torch.tensor(X_tr, dtype=torch.float32)
        ytr_t = torch.tensor(y_tr, dtype=torch.float32)
        loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=128, shuffle=True)

        mlp.train()
        for _ in range(70):
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                crit(mlp(xb).squeeze(), yb).backward()
                opt.step()

        oof_mlp[val_idx] = mlp_predict_proba(mlp, X_val)
        test_mlp_folds.append(mlp_predict_proba(mlp, X_test_sc))

    # Meta Features
    meta_train = np.column_stack([oof_xgb, oof_mlp])
    meta_test  = np.column_stack([np.mean(test_xgb_folds, axis=0), np.mean(test_mlp_folds, axis=0)])

    meta_scaler = StandardScaler()
    meta_train_sc = meta_scaler.fit_transform(meta_train)
    meta_test_sc  = meta_scaler.transform(meta_test)

    # ── Meta Model : Logistic Regression
    meta_lr = LogisticRegression(random_state=SEED, max_iter=1000)
    meta_lr.fit(meta_train_sc, y_tv)
    pred_lr = meta_lr.predict(meta_test_sc)

    # Metrics
    def get_metrics(y_true, y_pred):
        return {
            "Accuracy" : accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall"   : recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1"       : f1_score(y_true, y_pred, average="macro", zero_division=0)
        }

    m_lr = get_metrics(y_test, pred_lr)

    results.append({
        "Dataset"  : ds_name,
        "LR_Acc"   : round(m_lr["Accuracy"],  4),
        "LR_Prec"  : round(m_lr["Precision"], 4),
        "LR_Rec"   : round(m_lr["Recall"],    4),
        "LR_F1"    : round(m_lr["F1"],        4),
    })



results_df = pd.DataFrame(results)

print("\n" + "="*65)
print("  STACKING ENSEMBLE — Logistic Regression Meta")
print("="*65)
print(f"\n  {'Dataset':<10} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>10} {'F1-Score':>10}")
print("  " + "-" * 46)
for _, r in results_df.iterrows():
    print(f"  {r['Dataset']:<10} {r['LR_Acc']:>10.4f} {r['LR_Prec']:>10.4f} "
          f"{r['LR_Rec']:>10.4f} {r['LR_F1']:>10.4f}")

results_df.to_csv("stacking_lr_results.csv", index=False)
print("\n  Saved → stacking_lr_results.csv")
print("="*65)


  STACKING ENSEMBLE — Logistic Regression Meta

  Dataset      Accuracy  Precision     Recall   F1-Score
  ----------------------------------------------
  LFI            0.9384     0.9118     0.9418     0.9250
  XSS            0.9980     0.9972     0.9985     0.9978
  SQL            0.9952     0.9908     0.9963     0.9935
  OSC            0.9864     0.9850     0.9878     0.9862

  Saved → stacking_lr_results.csv
